# University Data Pipeline — Step 1
## Parsing YSU Apify Crawl → Structured Course Dataset

**Thesis:** Curriculum–Industry Alignment in Armenian IT Education  
**Notebook purpose:** Load, inspect, parse, and compare YSU program data from Apify crawl results.  
**Output:** `data/processed/university/ysu_courses_parsed.csv` and `data/processed/university/missing_programs.csv`

---

## Setup

In [23]:
import json
import re
import csv
import pandas as pd
from pathlib import Path

# Project root (two levels up from this notebook)
PROJECT_ROOT = Path('../../')

APIFY_JSON    = PROJECT_ROOT / 'data/raw/university/apify/ysu_merged.json'
REFERENCE_CSV = PROJECT_ROOT / 'data/raw/university/reference_curriculum.csv'
OUT_DIR       = PROJECT_ROOT / 'data/processed/university'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Paths set up. Project root:', PROJECT_ROOT.resolve())

Paths set up. Project root: /Users/lianaaghamalyan/thesis_data


---
## Step 1 — Load Apify JSON and Inspect Structure

The Apify actor `webpage-to-markdown` was used to crawl 14 YSU educational program pages.  
Each page was converted to markdown format. This cell loads that output and inspects its structure.

In [24]:
with open(APIFY_JSON) as f:
    apify_data = json.load(f)

print(f'Type: {type(apify_data)}')
print(f'Number of crawled pages: {len(apify_data)}')
print(f'\nFields in each record: {list(apify_data[0].keys())}')

Type: <class 'list'>
Number of crawled pages: 22

Fields in each record: ['url', 'markdown']


In [25]:
# Show all crawled URLs and markdown length for each page
print('Crawled pages:\n')
for i, item in enumerate(apify_data):
    md_len = len(item.get('markdown') or '')
    print(f'  [{i:02d}] {item["url"]}')
    print(f'       markdown length: {md_len:,} characters')

Crawled pages:

  [00] https://ysu.am/en/faculty/78/educational-program-349/edu-plan
       markdown length: 94,825 characters
  [01] https://www.ysu.am/en/faculty/85/educational-program-305/edu-plan
       markdown length: 280,177 characters
  [02] https://www.ysu.am/en/faculty/78/educational-program-348/edu-plan
       markdown length: 107,229 characters
  [03] https://www.ysu.am/en/faculty/85/educational-program-306/edu-plan
       markdown length: 89,702 characters
  [04] https://www.ysu.am/en/faculty/78/educational-program-350/edu-plan
       markdown length: 139,368 characters
  [05] https://www.ysu.am/en/faculty/85/educational-program-307/edu-plan
       markdown length: 86,262 characters
  [06] https://www.ysu.am/en/faculty/78/educational-program-351/edu-plan
       markdown length: 315,316 characters
  [07] https://www.ysu.am/en/faculty/85/educational-program-308/edu-plan
       markdown length: 90,940 characters
  [08] https://www.ysu.am/en/faculty/85/educational-program-309/

In [26]:
# Preview the header section (first 1500 chars) of the first page
# to understand what metadata is available before the course tables
print('=== Preview: First Page Header ===')
print(apify_data[1]['markdown'][:1500])

=== Preview: First Page Header ===
Title: Educational plan | YSU

URL Source: https://www.ysu.am/en/faculty/85/educational-program-305/edu-plan

Published Time: Fri, 20 Mar 2026 23:28:17 GMT

Markdown Content:
[](https://www.ysu.am/en/faculty/85/educational-program-305/edu-plan)

Type:

Bachelor

Speciality:

061101.02.6 - Ինֆորմատիկա (Համակարգչային գիտություն)

Qualification awarded:

Ինֆորմատիկայի բակալավր

Programme academic year:

2025/2026

Mode of study:

Full time

Language of study:

Հայերեն

### General educational component

| Chair code | Name of the course | Credits |
| --- | --- | --- |
| 1803 | Թուրքերեն-2 | 2 |
| 2-րդ ՝ գարնանային կիսամյակ 2 ժամ/շաբ գործ.-2 MANDATORY 1803/Բ02 1. Purpose of the Course · սովորեցնել ժամանակակից գրական թուրքերեն, · ձևավորել թուրքերեն տեքստեր վերարատադրելու ունակություններ, · զարգացնել հայերենից թուրքերեն և թուրքերենից հայերեն տեքստեր թարգմանելու հմտությունները: 2. Educational Outcomes ա. մասնագիտական գիտելիք և իմացություն 1. ներկայացնելու թո

### Observations — Apify JSON Structure

Each record has exactly **two fields**:

| Field | Description |
|---|---|
| `url` | URL of the crawled YSU program page |
| `markdown` | Full page content as markdown text |

The markdown of each page contains:
1. **Page metadata** (Type: Bachelor/Master, Speciality code, Qualification, Academic year, Mode, Language)
2. **Course tables** per curriculum component (General, Core, Elective, etc.), in markdown table format
3. **Detailed course descriptions** embedded after each table row (Armenian text)

The pages are from two faculties:
- Faculty **78** → programs 349–354
- Faculty **85** → programs 305–311

Pages with very short markdown (~23k chars) appear to be empty or minimal program pages.

---
## Step 2 — Parse Markdown → Structured Course-Level Dataset

We extract from each page:
- Program metadata (type, speciality, academic year)
- Course rows from all markdown tables (chair code, course name, credits)
- Component name (General / Core / Elective)

The result will be one row per course.

In [27]:
def extract_metadata(markdown: str) -> dict:
    """
    Extract program-level metadata from the markdown header.
    Returns a dict with keys: degree_type, speciality, qualification, academic_year, mode, language.
    """
    meta = {}

    def _find(label):
        # Match label followed by a value on the next non-empty line
        pattern = rf'{re.escape(label)}\s*\n+([^\n]+)'
        m = re.search(pattern, markdown)
        return m.group(1).strip() if m else None

    meta['degree_type']    = _find('Type:')
    meta['speciality']     = _find('Speciality:')
    meta['qualification']  = _find('Qualification awarded:')
    meta['academic_year']  = _find('Programme academic year:')
    meta['mode']           = _find('Mode of study:')
    meta['language']       = _find('Language of study:')

    return meta


def extract_courses(markdown: str, source_url: str) -> list:
    """
    Extract all course rows from all markdown tables in the page.
    Returns a list of dicts, one per course row.
    """
    meta = extract_metadata(markdown)
    courses = []

    # Find all component headings (### General / ### Core / ### Elective...)
    # Split the markdown into sections by component
    component_pattern = re.compile(r'###\s+([A-Za-z][^\n]+)\n', re.IGNORECASE)
    component_matches = list(component_pattern.finditer(markdown))

    # Build (component_name, section_text) pairs
    sections = []
    for i, m in enumerate(component_matches):
        comp_name = m.group(1).strip()
        start = m.end()
        end = component_matches[i + 1].start() if i + 1 < len(component_matches) else len(markdown)
        sections.append((comp_name, markdown[start:end]))

    # If no component headers found, treat whole markdown as one section
    if not sections:
        sections = [('Unknown', markdown)]

    # Table row pattern: | <chair_code> | <course_name> | <credits> |
    # First data row after the header separator
    table_row = re.compile(
        r'^\|\s*(\d+)\s*\|\s*([^|]+?)\s*\|\s*(\d*)\s*\|',
        re.MULTILINE
    )

    for comp_name, section_text in sections:
        for m in table_row.finditer(section_text):
            chair_code  = m.group(1).strip()
            course_name = m.group(2).strip()
            credits     = m.group(3).strip() or None

            courses.append({
                'source_url':     source_url,
                'degree_type':    meta.get('degree_type'),
                'speciality':     meta.get('speciality'),
                'academic_year':  meta.get('academic_year'),
                'component':      comp_name,
                'chair_code':     chair_code,
                'course_name_original': course_name,
                'credits':        credits,
            })

    return courses


print('Parsing functions defined.')

Parsing functions defined.


In [28]:
# Run the parser on all 14 pages
all_courses = []
page_summaries = []

for item in apify_data:
    url      = item['url']
    markdown = item.get('markdown') or ''
    courses  = extract_courses(markdown, url)
    meta     = extract_metadata(markdown)

    all_courses.extend(courses)
    page_summaries.append({
        'url':         url,
        'degree_type': meta.get('degree_type'),
        'speciality':  meta.get('speciality'),
        'n_courses':   len(courses),
    })

print(f'Total courses extracted: {len(all_courses)}')
print()

# Show per-page summary
summary_df = pd.DataFrame(page_summaries)
summary_df

Total courses extracted: 1131



,url,degree_type,speciality,n_courses
0,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,26
1,https://www.ysu.am/en/faculty/85/educational-p...,Bachelor,061101.02.6 - Ինֆորմատիկա (Համակարգչային գիտու...,89
2,https://www.ysu.am/en/faculty/78/educational-p...,Master,041301.04.7 - Կառավարում,26
3,https://www.ysu.am/en/faculty/85/educational-p...,Master,061101.04.7 - Ինֆորմատիկա (Համակարգչային գիտու...,36
4,https://www.ysu.am/en/faculty/78/educational-p...,Master,041201.01.7 - Finance,36
5,https://www.ysu.am/en/faculty/85/educational-p...,Master,061101.05.7 - Ինֆորմատիկա (Համակարգչային գիտու...,36
6,https://www.ysu.am/en/faculty/78/educational-p...,Bachelor,041301.01.6 - Կառավարում,83
7,https://www.ysu.am/en/faculty/85/educational-p...,Master,061105.02.7 - Տեղեկատվական տեխնոլոգիաներ,36
8,https://www.ysu.am/en/faculty/85/educational-p...,Bachelor,061101.02.6 - Ինֆորմատիկա (Համակարգչային գիտու...,95
9,https://www.ysu.am/en/faculty/85/educational-p...,None,None,0


In [29]:
# Build the courses DataFrame
courses_df = pd.DataFrame(all_courses)
print(f'Shape: {courses_df.shape}')
print(f'\nColumns: {list(courses_df.columns)}')
courses_df.head(10)

Shape: (1131, 8)

Columns: ['source_url', 'degree_type', 'speciality', 'academic_year', 'component', 'chair_code', 'course_name_original', 'credits']


,source_url,degree_type,speciality,academic_year,component,chair_code,course_name_original,credits
0,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,General educational component,1002,Տեղեկատվական տեխնոլոգիաները մասնագիտական ոլորտ...,3
1,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,General educational component,1002,Ռազմավարական պլանավորում,3
2,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,Professional educational component,1002,Տվյալների վերլուծության մաթեմատիկական մեթոդներ...,6
3,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,Professional educational component,1002,Մաթեմատիկա և հավանականությունների տեսություն տ...,6
4,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,Professional educational component,1002,Բիզնես գործընթացները և տվյալների վիզուալիզացիա,3
5,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,Professional educational component,1002,Տվյալների բազաների կառավարում,6
6,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,Professional educational component,1002,Data Mining տեխնոլոգիա,3
7,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,Professional educational component,1002,Մեքենայական ուսուցում,6
8,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,Professional educational component,1002,Նեյրոնային ցանցեր և խորը ուսուցում,6
9,https://ysu.am/en/faculty/78/educational-progr...,Master,031101.19.7 - Տնտեսագիտություն,2025/2026,Professional educational component,1002,Տվյալների շտեմարաններ և խողովակագիծ (Data Ware...,6


In [30]:
# Breakdown by degree type and component
print('Courses by degree type:')
print(courses_df['degree_type'].value_counts())
print()
print('Courses by component:')
print(courses_df['component'].value_counts())

Courses by degree type:
degree_type
Bachelor    811
Master      320
Name: count, dtype: int64

Courses by component:
component
Professional educational component    560
General educational component         517
Other educational modules              54
Name: count, dtype: int64


In [31]:
# Pages with very few or zero courses — likely empty program pages
print('Pages with 0 or very few courses parsed:')
print(summary_df[summary_df['n_courses'] < 5])

Pages with 0 or very few courses parsed:
                                                  url degree_type speciality  \
9   https://www.ysu.am/en/faculty/85/educational-p...        None       None   
12  https://www.ysu.am/en/faculty/78/educational-p...        None       None   
13  https://www.ysu.am/en/faculty/85/educational-p...        None       None   

    n_courses  
9           0  
12          0  
13          0  


In [32]:
# Save parsed courses to processed folder
out_path = OUT_DIR / 'ysu_courses_parsed.csv'
courses_df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}  ({len(courses_df)} rows)')

Saved: ../../data/processed/university/ysu_courses_parsed.csv  (1131 rows)


---
## Step 3 — Compare with Reference Curriculum Dataset

The reference dataset (`reference_curriculum.csv`) was manually collected and covers  
4 universities and 23 programs. We now compare it with the Apify-parsed YSU data to:
- Identify which programs are covered by both sources
- Find programs in the reference that were not crawled
- Find URLs in the reference that differ from crawled URLs

In [33]:
ref_df = pd.read_csv(REFERENCE_CSV)
print(f'Reference dataset shape: {ref_df.shape}')
print(f'Columns: {list(ref_df.columns)}')
ref_df.head(5)

Reference dataset shape: (1143, 14)
Columns: ['university', 'program_name', 'degree_level', 'component', 'course_name_original', 'course_name_english', 'chair_code', 'credits', 'hours_per_week', 'semester', 'study_year', 'program_duration', 'course_description', 'source_url']


,university,program_name,degree_level,component,course_name_original,course_name_english,chair_code,credits,hours_per_week,semester,study_year,program_duration,course_description,source_url
0,Yerevan State University,Informatics and Applied Mathematics,Bachelor,General,Թուրքերեն-2,Turkish-2,1803,2.0,NaN,NaN,NaN,NaN,NaN,https://ysu.am/en/faculty/85/educational-progr...
1,Yerevan State University,Informatics and Applied Mathematics,Bachelor,General,Փիլիսոփայություն,Philosophy,1311,3.0,NaN,NaN,NaN,NaN,NaN,https://ysu.am/en/faculty/85/educational-progr...
2,Yerevan State University,Informatics and Applied Mathematics,Bachelor,General,Էկոլոգիայի և բնապահպանության հիմունքներ,Ecology and Environmental Protection Fundamentals,708,2.0,NaN,NaN,NaN,NaN,NaN,https://ysu.am/en/faculty/85/educational-progr...
3,Yerevan State University,Informatics and Applied Mathematics,Bachelor,General,Ռուսաց լեզու-2,Russian Language-2,1705,2.0,NaN,NaN,NaN,NaN,NaN,https://ysu.am/en/faculty/85/educational-progr...
4,Yerevan State University,Informatics and Applied Mathematics,Bachelor,General,Հայոց պատմություն 2,Armenian History 2,1106,2.0,NaN,NaN,NaN,NaN,NaN,https://ysu.am/en/faculty/85/educational-progr...


In [34]:
# Overview of reference dataset
print('Universities in reference:')
print(ref_df['university'].value_counts())
print()
print('Programs in reference:')
print(ref_df['program_name'].value_counts())
print()
print('Degree levels:')
print(ref_df['degree_level'].value_counts())

Universities in reference:
university
Yerevan State University                                           781
National University of Architecture and Construction of Armenia    208
American University of Armenia                                     133
Russian-Armenian University                                         21
Name: count, dtype: int64

Programs in reference:
program_name
Applied Statistics and Data Science                                                             128
Radiophysics and Computer Technology                                                            103
Informatics and Applied Mathematics (Part time)                                                  96
Informatics and Applied Mathematics                                                              92
Data Processing in Physics and Artificial Intelligence                                           91
Information Security                                                                             91
Informatics (Co

In [35]:
# YSU-only reference data
ysu_ref = ref_df[ref_df['university'] == 'Yerevan State University'].copy()
print(f'YSU rows in reference: {len(ysu_ref)}')
print(f'YSU programs:')
print(ysu_ref[['program_name', 'degree_level', 'source_url']].drop_duplicates().to_string())

YSU rows in reference: 781
YSU programs:
                                                                                     program_name degree_level                                                      source_url
0                                                             Informatics and Applied Mathematics     Bachelor   https://ysu.am/en/faculty/85/educational-program-305/edu-plan
92                                                Informatics and Applied Mathematics (Part time)     Bachelor   https://ysu.am/en/faculty/85/educational-program-309/edu-plan
188                                                                          Information Security     Bachelor   https://ysu.am/en/faculty/85/educational-program-304/edu-plan
279                                              Discrete Mathematics and Theoretical Informatics       Master   https://ysu.am/en/faculty/85/educational-program-306/edu-plan
311                                                 Numerical Analysis and Mathemati

In [36]:
# Unique URLs in reference (YSU)
ref_ysu_urls = set(ysu_ref['source_url'].dropna().unique())
print(f'Unique source URLs in YSU reference: {len(ref_ysu_urls)}')
for url in sorted(ref_ysu_urls):
    print(' ', url)

Unique source URLs in YSU reference: 14
  https://ysu.am/en/faculty/516/educational-program-299/edu-plan
  https://ysu.am/en/faculty/516/educational-program-303/edu-plan
  https://ysu.am/en/faculty/516/educational-program-957/edu-plan
  https://ysu.am/en/faculty/520/educational-program-500/edu-plan
  https://ysu.am/en/faculty/520/educational-program-689/edu-plan
  https://ysu.am/en/faculty/525/educational-program-313/edu-plan
  https://ysu.am/en/faculty/525/educational-program-858/edu-plan
  https://ysu.am/en/faculty/78/educational-program-349/edu-plan
  https://ysu.am/en/faculty/85/educational-program-304/edu-plan
  https://ysu.am/en/faculty/85/educational-program-305/edu-plan
  https://ysu.am/en/faculty/85/educational-program-306/edu-plan
  https://ysu.am/en/faculty/85/educational-program-307/edu-plan
  https://ysu.am/en/faculty/85/educational-program-308/edu-plan
  https://ysu.am/en/faculty/85/educational-program-309/edu-plan


In [37]:
# Normalize URLs for comparison (strip trailing slashes, lowercase)
def normalize_url(url):
    if not url:
        return ''
    url = url.strip().rstrip('/')
    # Normalize www vs non-www
    url = url.replace('https://www.ysu.am', 'https://ysu.am')
    url = url.replace('http://www.ysu.am', 'https://ysu.am')
    url = url.replace('http://ysu.am', 'https://ysu.am')
    return url.lower()

crawled_urls_norm = set(normalize_url(item['url']) for item in apify_data)
ref_ysu_urls_norm = set(normalize_url(u) for u in ref_ysu_urls)

print(f'Normalized crawled URLs:  {len(crawled_urls_norm)}')
print(f'Normalized reference URLs (YSU): {len(ref_ysu_urls_norm)}')

in_both    = crawled_urls_norm & ref_ysu_urls_norm
only_crawl = crawled_urls_norm - ref_ysu_urls_norm
only_ref   = ref_ysu_urls_norm - crawled_urls_norm

print(f'\nURLs in BOTH crawl and reference: {len(in_both)}')
print(f'URLs ONLY in crawl (not in reference): {len(only_crawl)}')
print(f'URLs ONLY in reference (not crawled): {len(only_ref)}')

Normalized crawled URLs:  22
Normalized reference URLs (YSU): 14

URLs in BOTH crawl and reference: 14
URLs ONLY in crawl (not in reference): 8
URLs ONLY in reference (not crawled): 0


In [38]:
if in_both:
    print('=== Matched URLs (in both) ===')
    for u in sorted(in_both):
        print(' ', u)

if only_crawl:
    print('\n=== Crawled but NOT in reference ===')
    for u in sorted(only_crawl):
        print(' ', u)

if only_ref:
    print('\n=== In reference but NOT crawled ===')
    for u in sorted(only_ref):
        print(' ', u)

=== Matched URLs (in both) ===
  https://ysu.am/en/faculty/516/educational-program-299/edu-plan
  https://ysu.am/en/faculty/516/educational-program-303/edu-plan
  https://ysu.am/en/faculty/516/educational-program-957/edu-plan
  https://ysu.am/en/faculty/520/educational-program-500/edu-plan
  https://ysu.am/en/faculty/520/educational-program-689/edu-plan
  https://ysu.am/en/faculty/525/educational-program-313/edu-plan
  https://ysu.am/en/faculty/525/educational-program-858/edu-plan
  https://ysu.am/en/faculty/78/educational-program-349/edu-plan
  https://ysu.am/en/faculty/85/educational-program-304/edu-plan
  https://ysu.am/en/faculty/85/educational-program-305/edu-plan
  https://ysu.am/en/faculty/85/educational-program-306/edu-plan
  https://ysu.am/en/faculty/85/educational-program-307/edu-plan
  https://ysu.am/en/faculty/85/educational-program-308/edu-plan
  https://ysu.am/en/faculty/85/educational-program-309/edu-plan

=== Crawled but NOT in reference ===
  https://ysu.am/en/faculty/

---
## Step 4 — Identify Missing Source URLs and Programs

Here we build a comprehensive list of programs/URLs that need attention:
1. Reference programs with missing or empty `source_url`
2. Reference YSU programs whose URL was not included in the crawl
3. Non-YSU universities in reference (AUA, RAU, NUACA) — not yet crawled at all

In [39]:
# 1. Programs with missing source_url in reference
missing_url_rows = ref_df[ref_df['source_url'].isna() | (ref_df['source_url'].str.strip() == '')]
print(f'Rows with missing source_url: {len(missing_url_rows)}')
if len(missing_url_rows) > 0:
    missing_url_rows[['university', 'program_name', 'degree_level']].drop_duplicates()

Rows with missing source_url: 0


In [40]:
# 2. YSU programs in reference whose URL was NOT crawled
uncrawled_ysu_records = ysu_ref[
    ysu_ref['source_url'].apply(normalize_url).isin(only_ref)
][['university', 'program_name', 'degree_level', 'source_url']].drop_duplicates()

print(f'YSU programs in reference but NOT crawled: {len(uncrawled_ysu_records)}')
uncrawled_ysu_records

YSU programs in reference but NOT crawled: 0


,university,program_name,degree_level,source_url


In [41]:
# 3. Non-YSU universities — not crawled at all yet
other_unis = ref_df[ref_df['university'] != 'Yerevan State University']
other_unis_summary = other_unis.groupby(
    ['university', 'program_name', 'degree_level', 'source_url']
).size().reset_index(name='course_count').sort_values(['university', 'program_name'])

print(f'Non-YSU universities in reference (not yet crawled with Apify):')
other_unis_summary

Non-YSU universities in reference (not yet crawled with Apify):


,university,program_name,degree_level,source_url,course_count
0,American University of Armenia,Computer Science,Bachelor,https://cse.aua.am/cs/,50
1,American University of Armenia,Computer and Information Science,Master,http://cse.aua.am/graduate-programs/cis/,27
2,American University of Armenia,Data Science,Bachelor,https://cse.aua.am/ds/,33
3,American University of Armenia,Industrial Engineering and Systems Management,Master,https://cse.aua.am/iesm-home/,23
4,National University of Architecture and Constr...,GEOGRAPHIC INFORMATION SYSTEMS,Master,https://nuaca.am/archives/144938/?lang=en,21
5,National University of Architecture and Constr...,INFORMATICS (COMPUTER SCIENCE),Master,https://nuaca.am/archives/144925/?lang=en,23
6,National University of Architecture and Constr...,Informatics (Computer Science),Bachelor,https://nuaca.am/archives/145834/?lang=en,70
7,National University of Architecture and Constr...,Information systems,Bachelor,https://nuaca.am/archives/145838/?lang=en,69
8,National University of Architecture and Constr...,PROJECT MANAGEMENT (in Information Technologies),Master,https://nuaca.am/archives/144950/?lang=en,25
9,Russian-Armenian University,Applied Mathematics and Informatics,Bachelor,https://impht.rau.am/am/program/bachelor,21


In [42]:
# 4. Crawled URLs not yet in reference (new data from Apify, need to be added to reference)
crawled_only_df = pd.DataFrame([
    {'url': item['url'],
     'degree_type': extract_metadata(item.get('markdown') or '').get('degree_type'),
     'speciality':  extract_metadata(item.get('markdown') or '').get('speciality'),
     'n_courses': len(extract_courses(item.get('markdown') or '', item['url']))}
    for item in apify_data
    if normalize_url(item['url']) in only_crawl
])

print(f'URLs crawled but not in reference dataset:')
crawled_only_df

URLs crawled but not in reference dataset:


,url,degree_type,speciality,n_courses
0,https://www.ysu.am/en/faculty/78/educational-p...,Master,041301.04.7 - Կառավարում,26
1,https://www.ysu.am/en/faculty/78/educational-p...,Master,041201.01.7 - Finance,36
2,https://www.ysu.am/en/faculty/78/educational-p...,Bachelor,041301.01.6 - Կառավարում,83
3,https://www.ysu.am/en/faculty/85/educational-p...,None,None,0
4,https://www.ysu.am/en/faculty/78/educational-p...,Bachelor,031101.01.6 - Տնտեսագիտություն,83
5,https://www.ysu.am/en/faculty/78/educational-p...,Bachelor,041201.01.6 - Ֆինանսներ,83
6,https://www.ysu.am/en/faculty/78/educational-p...,None,None,0
7,https://www.ysu.am/en/faculty/85/educational-p...,None,None,0


---
## Step 5 — Output: missing_programs.csv

We consolidate all gaps into one actionable file:  
`data/processed/university/missing_programs.csv`

This file captures:
- YSU programs in reference but not crawled → need to be scraped
- Non-YSU university programs → need new Apify runs
- Programs with empty `source_url` → need manual URL lookup

In [43]:
missing_records = []

# Category A: YSU in reference but not crawled
for _, row in uncrawled_ysu_records.iterrows():
    missing_records.append({
        'university':    row['university'],
        'program_name':  row['program_name'],
        'degree_level':  row['degree_level'],
        'source_url':    row['source_url'],
        'reason':        'In reference CSV but not in Apify crawl',
        'action':        'Re-scrape with Apify'
    })

# Category B: Missing source_url
for _, row in missing_url_rows[['university','program_name','degree_level','source_url']].drop_duplicates().iterrows():
    missing_records.append({
        'university':    row['university'],
        'program_name':  row['program_name'],
        'degree_level':  row['degree_level'],
        'source_url':    '',
        'reason':        'Missing source_url in reference CSV',
        'action':        'Find URL manually, then scrape'
    })

# Category C: Other universities not yet crawled
for _, row in other_unis_summary.iterrows():
    missing_records.append({
        'university':    row['university'],
        'program_name':  row['program_name'],
        'degree_level':  row['degree_level'],
        'source_url':    row['source_url'],
        'reason':        'University not yet crawled with Apify',
        'action':        'Create new Apify run for this university'
    })

missing_df = pd.DataFrame(missing_records).drop_duplicates(subset=['university','program_name','degree_level','source_url'])
print(f'Total missing/incomplete entries: {len(missing_df)}')
missing_df

Total missing/incomplete entries: 10


,university,program_name,degree_level,source_url,reason,action
0,American University of Armenia,Computer Science,Bachelor,https://cse.aua.am/cs/,University not yet crawled with Apify,Create new Apify run for this university
1,American University of Armenia,Computer and Information Science,Master,http://cse.aua.am/graduate-programs/cis/,University not yet crawled with Apify,Create new Apify run for this university
2,American University of Armenia,Data Science,Bachelor,https://cse.aua.am/ds/,University not yet crawled with Apify,Create new Apify run for this university
3,American University of Armenia,Industrial Engineering and Systems Management,Master,https://cse.aua.am/iesm-home/,University not yet crawled with Apify,Create new Apify run for this university
4,National University of Architecture and Constr...,GEOGRAPHIC INFORMATION SYSTEMS,Master,https://nuaca.am/archives/144938/?lang=en,University not yet crawled with Apify,Create new Apify run for this university
5,National University of Architecture and Constr...,INFORMATICS (COMPUTER SCIENCE),Master,https://nuaca.am/archives/144925/?lang=en,University not yet crawled with Apify,Create new Apify run for this university
6,National University of Architecture and Constr...,Informatics (Computer Science),Bachelor,https://nuaca.am/archives/145834/?lang=en,University not yet crawled with Apify,Create new Apify run for this university
7,National University of Architecture and Constr...,Information systems,Bachelor,https://nuaca.am/archives/145838/?lang=en,University not yet crawled with Apify,Create new Apify run for this university
8,National University of Architecture and Constr...,PROJECT MANAGEMENT (in Information Technologies),Master,https://nuaca.am/archives/144950/?lang=en,University not yet crawled with Apify,Create new Apify run for this university
9,Russian-Armenian University,Applied Mathematics and Informatics,Bachelor,https://impht.rau.am/am/program/bachelor,University not yet crawled with Apify,Create new Apify run for this university


In [44]:
# Save to processed folder
out_missing = OUT_DIR / 'missing_programs.csv'
missing_df.to_csv(out_missing, index=False, encoding='utf-8-sig')
print(f'Saved: {out_missing}  ({len(missing_df)} rows)')

Saved: ../../data/processed/university/missing_programs.csv  (10 rows)


---
## Summary

| Output | Location | Description |
|---|---|---|
| `ysu_courses_parsed.csv` | `data/processed/university/` | Course-level dataset parsed from Apify JSON (22 YSU pages) |
| `missing_programs.csv` | `data/processed/university/` | All programs/URLs that need attention before next pipeline step |

### Next Steps
1. Review `missing_programs.csv` and run new Apify scrapes for missing programs
2. Add English translations for Armenian course names in `ysu_courses_parsed.csv`
3. Merge Apify-parsed courses with reference CSV into one unified course dataset
4. Proceed to NLP skill extraction (`02_skill_extraction.ipynb`)